# C44 convergence check — were the from-scratch arms under-trained?

**Runtime: L4 GPU + High-RAM.** ~8 h, fully resumable. If the runtime drops, re-run the sweep
cell and it continues from where it stopped.

Run the three cells in order. The last cell prints the verdict for you — you do not have to
interpret anything.

---

## What this decides

C44 reports a **5.2 accuracy-point** AG News gap between a 99M from-scratch encoder (0.8878) and
pretrained BERT (0.9396), at 3 epochs. That gap is the **positive control** for the pretraining
instrument: it is the evidence the instrument can detect a real pretraining benefit, which is what
licenses reading the clinical result (C37: gap +0.0031, p = 0.690) as a genuine null rather than as
an experiment too insensitive to see anything.

**The threat.** Every arm trained for exactly 3 epochs with no per-arm tuning, and the 100M arm
gained only +0.009 over the 10M arm. If the scratch arms had simply not finished learning, the
5.2-point gap measures *training budget*, not pretraining, and the control collapses.

## Design

A **budget sweep**, not early stopping. Each run gets its own complete OneCycleLR schedule over its
own epoch budget, so no run is ever cut off mid-schedule — which is what makes the 3-epoch point an
exact reproduction of the original protocol and the trend across budgets interpretable.

Budgets **3, 6, 12, 24** × 4 arms × 3 seeds. Both scratch AND BERT get every budget: giving scratch
extra epochs alone would replace one bias with another. Model selection is on a held-out validation
split drawn from AG News train rows the original sweep never touched, so the 40,000-row training set
is byte-identical and the two studies stay comparable. Test is never used for selection.

Public data only (AG News). No MIMIC, no credentialed data anywhere in this notebook.

## The decision rule — committed before this was ever run

Full text: `accord/scaling/C44_PREREGISTRATION.md`, committed at `b048829`. I verified no
convergence results existed locally or on Drive before writing it.

This matters because this project has retracted five mechanism claims in two weeks, every one of
them written *after* the numbers were seen. The one time a kill condition was committed in advance,
it fired against the claim its author wanted and that was discovered in a day rather than at a viva.

Read the gap between `scratch_100M` and `pretrained_bert` at the **24-epoch** budget:

| outcome | condition | consequence |
|---|---|---|
| **A — control holds** | gap **≥ 3.0 pts** AND ±1 SD bands do not overlap | C44 stands. The pretraining chapter stands at ~320 lines. C37's clinical result is reportable as a genuine null. |
| **B — control fails** | gap **< 3.0 pts** OR ordering not seed-resolved | C44 withdrawn. C37 reverts to "too insensitive to see anything." Chapter collapses to ~140 lines and merges. |
| **INCONCLUSIVE** | gap in **[2.5, 3.5]** or SD bands touch | Report the budget curve as the finding; write at Outcome B length. |

**Outcome B is not a failure of the thesis.** An instrument whose positive control does not hold is
a result — it is the case where adding the control revealed the original reading was never
interpretable. That is the same shape as C52/C56.

**Explicitly forbidden:** resolving an INCONCLUSIVE result by adding seeds until it moves. That is
optional stopping, and it is how C47 happened.

### 1. Install

In [ ]:
!pip -q install -U datasets transformers scikit-learn

### 2. Everything — setup, data, models, and the resumable sweep

In [ ]:
import os, json, time, math, statistics as st
import numpy as np, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

assert torch.cuda.is_available(), "No GPU -- Runtime > Change runtime type > L4 GPU"
print(f"device: {torch.cuda.get_device_name(0)}   torch {torch.__version__}")

from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/thesis/results_2026-08/scaling'
os.makedirs(OUT, exist_ok=True)
RESULTS_PATH = f"{OUT}/convergence_results.json"
print("results ->", RESULTS_PATH)

# ---- data -------------------------------------------------------------------
# tr_idx reproduces the original sweep exactly: same seed, same call, same order.
# The validation split is drawn from indices the original run did not touch, so
# the 40,000 training rows are unchanged and the two studies stay comparable.
ds  = load_dataset("fancyzhx/ag_news")
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
MAXLEN, N_TRAIN, N_VAL = 256, 40_000, 5_000

rng    = np.random.default_rng(42)
tr_idx = rng.choice(len(ds['train']), N_TRAIN, replace=False)
held   = np.setdiff1d(np.arange(len(ds['train'])), tr_idx)
val_idx = np.random.default_rng(43).choice(held, N_VAL, replace=False)
assert len(np.intersect1d(tr_idx, val_idx)) == 0, "train/val overlap"

def encode(split, idx=None):
    d = split.select(idx) if idx is not None else split
    e = tok(list(d['text']), truncation=True, padding='max_length',
            max_length=MAXLEN, return_tensors='pt')
    return e['input_ids'], e['attention_mask'], torch.tensor(list(d['label']))

Xtr, Mtr, ytr = encode(ds['train'], tr_idx)
Xva, Mva, yva = encode(ds['train'], val_idx)
Xte, Mte, yte = encode(ds['test'])
print(f"train {tuple(Xtr.shape)}  val {tuple(Xva.shape)}  test {tuple(Xte.shape)}")

# ---- models -- identical definitions to the original sweep -------------------
class Enc(nn.Module):
    """Pre-LN transformer encoder. Non-embedding params ~ 12*d^2*L."""
    def __init__(self, vocab, d, L, heads, n_cls=4, maxlen=MAXLEN):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(maxlen, d)
        layer = nn.TransformerEncoderLayer(d, heads, 4*d, 0.1, batch_first=True,
                                           norm_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(layer, L)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_cls)
    def forward(self, ids, mask):
        p = torch.arange(ids.size(1), device=ids.device)
        h = self.enc(self.emb(ids) + self.pos(p), src_key_padding_mask=~mask.bool())
        h = (self.norm(h) * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
        return self.head(h)

class Wrap(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, ids, mask):
        return self.m(input_ids=ids, attention_mask=mask).logits

def non_emb(m):
    e = sum(p.numel() for n, p in m.named_parameters()
            if 'embedding' in n.lower() or n.startswith(('emb.', 'pos.')))
    return sum(p.numel() for p in m.parameters()) - e

CONFIGS = {'10M': (384, 6, 6), '30M': (576, 8, 8), '100M': (768, 14, 12)}
BUDGETS = [3, 6, 12, 24]
SEEDS   = [0, 1, 2]

# ---- train / evaluate --------------------------------------------------------
@torch.no_grad()
def evaluate(model, X, M, y, dev='cuda'):
    model.eval(); preds = []
    with torch.autocast('cuda', dtype=torch.float16):
        for i in range(0, len(X), 128):
            preds.append(model(X[i:i+128].to(dev), M[i:i+128].to(dev)).argmax(-1).cpu())
    p = torch.cat(preds)
    return float(accuracy_score(y, p)), float(f1_score(y, p, average='macro'))

def run(model, seed, tag, epochs, bs=32, lr=3e-4):
    """One complete OneCycle schedule over `epochs`. Validation accuracy is
    recorded every epoch so the shape of the curve is visible, not just its end."""
    torch.manual_seed(seed); np.random.seed(seed)
    dev = 'cuda'; model = model.to(dev)
    dl = DataLoader(TensorDataset(Xtr, Mtr, ytr), batch_size=bs, shuffle=True,
                    generator=torch.Generator().manual_seed(seed))
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs*len(dl))
    lossf = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda')
    hist, t0 = [], time.time()
    for ep in range(epochs):
        model.train(); run_loss, nb = 0.0, 0
        for x, m, y in dl:
            x, m, y = x.to(dev), m.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                loss = lossf(model(x, m), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item(); nb += 1
        va, _ = evaluate(model, Xva, Mva, yva)
        hist.append({"epoch": ep+1, "train_loss": run_loss/nb, "val_accuracy": va})
        print(f"  {tag} s{seed} E{epochs} ep{ep+1}/{epochs}  "
              f"loss {run_loss/nb:.4f}  val {va:.4f}", flush=True)
    te_acc, te_f1 = evaluate(model, Xte, Mte, yte)
    return {"tag": tag, "seed": seed, "budget": epochs,
            "non_emb_params": non_emb(model),
            "val_accuracy": hist[-1]["val_accuracy"],
            "accuracy": te_acc, "macro_f1": te_f1,
            "final_train_loss": hist[-1]["train_loss"],
            "history": hist, "minutes": (time.time()-t0)/60}

results = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else []
done = {(r['tag'], r['seed'], r['budget']) for r in results}
print(f"\n{len(done)} runs already complete -- resuming" if done else "\nstarting fresh")
def save(): json.dump(results, open(RESULTS_PATH, "w"), indent=2)

# ---- the sweep ---------------------------------------------------------------
# Budget is the outer loop so that if the runtime dies, every arm already has a
# complete set at the budgets finished so far -- a usable partial result rather
# than one arm swept and the rest missing.
for E in BUDGETS:
    for name, (d, L, h) in CONFIGS.items():
        for seed in SEEDS:
            tag = f"scratch_{name}"
            if (tag, seed, E) in done: continue
            r = run(Enc(tok.vocab_size, d, L, h), seed, tag, epochs=E)
            print(f"DONE {tag} s{seed} E{E}: test {r['accuracy']:.4f} ({r['minutes']:.1f}m)\n")
            results.append(r); save(); torch.cuda.empty_cache()
    for seed in SEEDS:
        tag = "pretrained_bert"
        if (tag, seed, E) in done: continue
        m = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)
        r = run(Wrap(m), seed, tag, epochs=E, lr=2e-5)
        print(f"DONE {tag} s{seed} E{E}: test {r['accuracy']:.4f} ({r['minutes']:.1f}m)\n")
        results.append(r); save(); torch.cuda.empty_cache()

print("convergence sweep complete")

### 3. Verdict — applies the committed rule automatically. Safe to re-run on a partial sweep.

In [ ]:
import json, statistics as st

results = json.load(open(RESULTS_PATH))
by = {}
for r in results:
    by.setdefault((r['tag'], r['budget']), []).append(r['accuracy'])

budgets = sorted({b for _, b in by})
arms = ['scratch_10M', 'scratch_30M', 'scratch_100M', 'pretrained_bert']

print(f"{'arm':<18}" + "".join(f"{f'E={b}':>17}" for b in budgets))
for a in arms:
    row = f"{a:<18}"
    for b in budgets:
        v = by.get((a, b))
        row += f"{(f'{st.mean(v):.4f}' + (f'±{st.stdev(v):.4f}' if len(v) > 1 else '')):>17}" if v else f"{'--':>17}"
    print(row)

print(f"\n{'budget':<9}{'scratch_100M':>16}{'BERT':>16}{'gap (pts)':>12}")
for b in budgets:
    s, p = by.get(('scratch_100M', b)), by.get(('pretrained_bert', b))
    if s and p:
        print(f"{b:<9}{st.mean(s):>16.4f}{st.mean(p):>16.4f}{(st.mean(p)-st.mean(s))*100:>12.2f}")

# ---- the pre-registered rule, applied verbatim -----------------------------
TARGET = 24
s, p = by.get(('scratch_100M', TARGET)), by.get(('pretrained_bert', TARGET))

print("\n" + "=" * 72)
if not (s and p):
    done = sorted({b for b in budgets if by.get(('scratch_100M', b)) and by.get(('pretrained_bert', b))})
    print(f"SWEEP INCOMPLETE — the {TARGET}-epoch budget has not finished.")
    print(f"Budgets complete so far: {done or 'none'}. Re-run the sweep cell; it resumes.")
    print("The rule is evaluated ONLY at 24 epochs. Do not read a verdict off a partial sweep.")
else:
    gap = (st.mean(p) - st.mean(s)) * 100
    sd_s = st.stdev(s) * 100 if len(s) > 1 else 0.0
    sd_p = st.stdev(p) * 100 if len(p) > 1 else 0.0
    resolved = (st.mean(p) - sd_p / 100) > (st.mean(s) + sd_s / 100)

    print(f"gap at {TARGET} epochs : {gap:.2f} accuracy points")
    print(f"scratch_100M        : {st.mean(s):.4f} ± {sd_s/100:.4f}")
    print(f"pretrained_bert     : {st.mean(p):.4f} ± {sd_p/100:.4f}")
    print(f"±1 SD bands disjoint: {resolved}")
    print("-" * 72)

    if 2.5 <= gap <= 3.5 or not resolved and gap >= 2.5:
        print("VERDICT: INCONCLUSIVE")
        print("  Report the budget curve as the finding. Write the chapter at Outcome B length.")
        print("  DO NOT add seeds to move it — that is optional stopping (see C47).")
    elif gap >= 3.0 and resolved:
        print("VERDICT: OUTCOME A — the positive control HOLDS")
        print("  C44 stands. The arms were converged; the gap is not a budget artifact.")
        print("  C37's clinical result is reportable as a genuine null.")
    else:
        print("VERDICT: OUTCOME B — the positive control FAILS")
        print("  C44 is withdrawn. C37 reverts to 'an experiment too insensitive to see anything.'")
        print("  This is a RESULT, not a setback: the control revealed the original reading")
        print("  was never interpretable. Same shape as C52/C56.")
print("=" * 72)
print(f"\nraw results saved to: {RESULTS_PATH}")
print("Tell Claude the verdict line and it will record the correction and update the chapter plan.")